# VisionCore: CIFAR-10 Dataset Exploration & Pipeline Foundations
### A PyTorch-Based Computer Vision Exploration and Analysis

This notebook explores the **CIFAR-10** (Canadian Institute For Advanced Research) dataset, inspecting its dimensional characteristics, class distributions, RGB channel statistics, normalization techniques, and architectural motivations for Convolutional Neural Networks (CNN) versus Artificial Neural Networks (ANN).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torchvision
import torchvision.transforms as transforms

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.utils import set_seed, get_class_emoji

set_seed(42)
sns.set_theme(style="whitegrid")
print(f"PyTorch Version: {torch.__version__}")
print(f"Device Detected: {config.DEVICE}")

## 1. Downloading & Inspecting CIFAR-10
The CIFAR-10 dataset consists of **60,000 32x32 colour images** in 10 classes, with 6,000 images per class.
- 50,000 training images
- 10,000 test images

In [ ]:
raw_train = torchvision.datasets.CIFAR10(
    root=str(config.DATA_DIR),
    train=True,
    download=True
)

raw_test = torchvision.datasets.CIFAR10(
    root=str(config.DATA_DIR),
    train=False,
    download=True
)

print(f"Training set samples: {len(raw_train):,}")
print(f"Test set samples:     {len(raw_test):,}")
print(f"Number of classes:    {len(raw_train.classes)}")
print(f"Classes: {raw_train.classes}")

## 2. Image Dimensions, Pixel Values & RGB Channels
- In NumPy / PIL: image shape is `(32, 32, 3)` -> `(Height, Width, Channels)`
- In PyTorch: image shape is `(3, 32, 32)` -> `(Channels, Height, Width)`
- **RGB Channels:**
  - **Red Channel:** Intensity of red light per pixel [0-255]
  - **Green Channel:** Intensity of green light per pixel [0-255]
  - **Blue Channel:** Intensity of blue light per pixel [0-255]

In [ ]:
sample_img, sample_label = raw_train[0]
sample_np = np.array(sample_img)

print(f"PIL Image Size:        {sample_img.size} (Width x Height)")
print(f"NumPy Array Shape:     {sample_np.shape} (Height x Width x Channels)")
print(f"Pixel Value Range:     Min = {sample_np.min()}, Max = {sample_np.max()}")
print(f"Data Type:             {sample_np.dtype}")
print(f"Ground Truth Class:    {sample_label} ({config.CLASS_NAMES[sample_label]})")

### Channel Separation Visualization
Let's visualize the three distinct color channels (Red, Green, Blue) of a sample image.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), dpi=150)

# Full RGB
axes[0].imshow(sample_np)
axes[0].set_title(f"Full RGB Image\n({config.CLASS_NAMES[sample_label].capitalize()})", fontweight="bold")
axes[0].axis("off")

# Red Channel
axes[1].imshow(sample_np[:, :, 0], cmap="Reds")
axes[1].set_title("Red Channel", fontweight="bold")
axes[1].axis("off")

# Green Channel
axes[2].imshow(sample_np[:, :, 1], cmap="Greens")
axes[2].set_title("Green Channel", fontweight="bold")
axes[2].axis("off")

# Blue Channel
axes[3].imshow(sample_np[:, :, 2], cmap="Blues")
axes[3].set_title("Blue Channel", fontweight="bold")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 3. Class Distribution Inspection
CIFAR-10 is a balanced dataset with exactly 5,000 training images per class (50,000 total) and 1,000 test images per class (10,000 total).

In [ ]:
from collections import Counter
train_counts = Counter(raw_train.targets)

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
classes = [config.CLASS_NAMES[i] for i in range(10)]
counts = [train_counts[i] for i in range(10)]

palette = sns.color_palette("viridis", 10)
bars = ax.bar(classes, counts, color=palette, edgecolor="black", alpha=0.85)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + 100, f"{int(yval):,}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_title("Class Distribution in CIFAR-10 Training Set", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Number of Images", fontsize=11, fontweight="bold")
ax.set_ylim(0, 6000)
plt.xticks(rotation=30, ha="right", fontsize=10)
plt.tight_layout()
plt.show()

## 4. Multi-Sample Grid Visualization with Class Labels

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(14, 9), dpi=150)
axes = axes.flatten()

np.random.seed(42)
sample_indices = np.random.choice(len(raw_train), size=24, replace=False)

for i, idx in enumerate(sample_indices):
    img, label = raw_train[idx]
    class_name = config.CLASS_NAMES[label]
    emoji = get_class_emoji(class_name)
    
    axes[i].imshow(img)
    axes[i].set_title(f"{emoji} {class_name.capitalize()}", fontsize=9, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("CIFAR-10 Random Dataset Exploration Grid", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 5. Normalization & Preprocessing Statistics
In deep learning with PyTorch, we normalize images using the dataset's channel-wise mean and standard deviation:
- `Mean: [0.4914, 0.4822, 0.4465]`
- `Std:  [0.2470, 0.2435, 0.2616]`

$$\text{Normalized Pixel} = \frac{\text{Pixel Value} - \mu_c}{\sigma_c}$$

This zero-centers each color channel, preventing gradient vanishing/explosion and ensuring balanced weight updates.

In [ ]:
# Demonstrating ToTensor + Normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=config.CIFAR10_MEAN, std=config.CIFAR10_STD)
])

tensor_img = transform(sample_img)
print(f"Transformed PyTorch Tensor Shape: {tensor_img.shape} -> (Channels, Height, Width)")
print(f"Normalized Min Value:            {tensor_img.min():.4f}")
print(f"Normalized Max Value:            {tensor_img.max():.4f}")
print(f"Normalized Mean per Channel:     {[round(m.item(), 4) for m in tensor_img.mean(dim=(1, 2))]}")

## 6. Architectural Comparison: Why CNN beats ANN for Computer Vision

| Feature | Baseline ANN (Fully Connected) | VisionCore CNN (Convolutional) |
| :--- | :--- | :--- |
| **Input Handling** | Flattens 2D image into 1D vector ($32 \times 32 \times 3 = 3072$) | Retains 2D grid structure $(3, 32, 32)$ |
| **Spatial Locality** | Completely destroyed upon flattening | Preserved through local receptive fields (kernels) |
| **Weight Sharing** | No (each connection has unique weight) | Yes (same 3x3 filter slides across image) |
| **Parameter Efficiency** | Highly inefficient, prone to overfitting | Highly efficient parameter reuse |
| **Translation Invariance** | None (pixel shift breaks classification) | Built-in via pooling and sliding convolution |
| **Feature Hierarchy** | Learns global pixel correlations | Learns hierarchical edges $\to$ textures $\to$ object parts |